``NumericalFeature.from_pssm`` turns position-specific scoring matrices (PSSMs) into a per-residue ``dict_num`` of ``(L, 20)`` arrays for :meth:`CPP.run_num`. It reads PSI-BLAST ASCII ``.pssm`` files (``psiblast -out_ascii_pssm``), reorders the PSI-BLAST column order ``ARNDCQEGHILKMFPSTWYV`` into the canonical ``ACDEFGHIKLMNPQRSTVWY``, and maps the values onto ``[0, 1]``.

We first write small synthetic PSSM files in the PSI-BLAST layout into a temporary folder (in real work, these come from PSI-BLAST):

In [1]:
import tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import aaanalysis as aa
aa.options["verbose"] = False

ORDER = "ARNDCQEGHILKMFPSTWYV"   # PSI-BLAST column order

def write_pssm(path, seq, rng):
    header = "".join(f"{a:>4}" for a in ORDER)
    lines = ["", "Last position-specific scoring matrix computed, weighted observed percentages rounded down, "
             "information per position, and relative weight of gapless real matches to pseudocounts",
             "           " + header + header]
    for i, res in enumerate(seq):
        log_odds = "".join(f"{s:4d}" for s in rng.integers(-4, 9, 20))
        percent = "".join(f"{p:4d}" for p in rng.integers(0, 40, 20))
        lines.append(f"{i + 1:5d} {res} {log_odds}  {percent}  0.45 0.12")
    lines += ["", "                      K         Lambda", "Standard Ungapped    0.1330     0.3176",
              "Standard Gapped      0.0410     0.2670", "PSI Ungapped         0.1420     0.3176",
              "PSI Gapped           0.0410     0.2670"]
    Path(path).write_text("\n".join(lines) + "\n")

df_seq = aa.load_dataset(name="DOM_GSEC", n=10)
pssm_dir = Path(tempfile.mkdtemp())
rng = np.random.default_rng(42)
for entry, seq in zip(df_seq["entry"], df_seq["sequence"]):
    write_pssm(pssm_dir / f"{entry}.pssm", seq, rng)

first_file = sorted(pssm_dir.glob("*.pssm"))[0]
print("\n".join(first_file.read_text().splitlines()[:6]))


Last position-specific scoring matrix computed, weighted observed percentages rounded down, information per position, and relative weight of gapless real matches to pseudocounts
              A   R   N   D   C   Q   E   G   H   I   L   K   M   F   P   S   T   W   Y   V   A   R   N   D   C   Q   E   G   H   I   L   K   M   F   P   S   T   W   Y   V
    1 M    5   4   1   3   6   0   1  -1  -2  -4  -4  -3   5   4  -3   5   5   7   8  -3    20  32   9  23   3  12  39  29  10  39   8  15  17  25  10  10  26   8   5   5  0.45 0.12
    2 G    0  -4  -4   5   3  -3  -1   4   6   7  -2   7   1  -1   3  -3   2   1   3  -2     2  14  39  22  21  25  20   8  39   9  25  34  26  37   7  25   3  22   7  26  0.45 0.12
    3 G    0  -3   7   7  -2  -2  -4   4   7  -2   1   5   1   5   5   5   5   1   1   3     4  23   1  28  24  19  12  20  25  27  36   6   1  26   6   5  10  28   2  15  0.45 0.12


Pass the folder as ``pssm`` (each file stem is the ``entry``). With ``df_seq``, every entry is checked for a PSSM whose row count and residues match its sequence:

In [2]:
nf = aa.NumericalFeature()
dict_num = nf.from_pssm(pssm=pssm_dir, df_seq=df_seq)

entry = df_seq["entry"][0]
df_pssm = pd.DataFrame(dict_num[entry], columns=aa.utils.LIST_CANONICAL_AA)   # canonical column order
aa.display_df(df_pssm, n_rows=10, show_shape=True)

DataFrame shape: (87, 20)


,A,C,D,E,F,G,H,I,K,L,M,N,P,Q,R,S,T,V,W,Y
1,0.047426,0.731059,0.731059,0.047426,0.993307,0.993307,0.119203,0.047426,0.999665,0.880797,0.993307,0.982014,0.993307,0.999089,0.997527,0.997527,0.880797,0.731059,0.047426,0.997527
2,0.119203,0.017986,0.500000,0.731059,0.880797,0.999089,0.982014,0.997527,0.119203,0.993307,0.500000,0.993307,0.880797,0.999665,0.993307,0.017986,0.952574,0.982014,0.119203,0.993307
3,0.999665,0.982014,0.997527,0.047426,0.731059,0.500000,0.993307,0.997527,0.997527,0.731059,0.997527,0.119203,0.999089,0.993307,0.731059,0.268941,0.268941,0.047426,0.982014,0.982014
4,0.731059,0.999089,0.880797,0.047426,0.952574,0.993307,0.952574,0.982014,0.952574,0.952574,0.047426,0.982014,0.997527,0.952574,0.982014,0.268941,0.952574,0.731059,0.017986,0.500000
5,0.999665,0.731059,0.731059,0.500000,0.993307,0.119203,0.500000,0.017986,0.047426,0.047426,0.997527,0.731059,0.993307,0.997527,0.982014,0.880797,0.993307,0.880797,0.119203,0.999089
6,0.500000,0.731059,0.993307,0.993307,0.731059,0.999665,0.268941,0.997527,0.993307,0.268941,0.997527,0.880797,0.993307,0.268941,0.999089,0.268941,0.047426,0.999089,0.047426,0.731059
7,0.047426,0.982014,0.982014,0.993307,0.500000,0.731059,0.997527,0.017986,0.880797,0.119203,0.119203,0.047426,0.982014,0.047426,0.952574,0.047426,0.982014,0.952574,0.047426,0.119203
8,0.268941,0.731059,0.880797,0.119203,0.500000,0.952574,0.952574,0.880797,0.268941,0.997527,0.999665,0.982014,0.952574,0.999665,0.880797,0.880797,0.999089,0.017986,0.731059,0.268941
9,0.500000,0.999665,0.999089,0.268941,0.997527,0.017986,0.047426,0.952574,0.500000,0.952574,0.500000,0.952574,0.731059,0.119203,0.047426,0.997527,0.999665,0.999665,0.500000,0.119203
10,0.997527,0.047426,0.999089,0.880797,0.500000,0.500000,0.017986,0.997527,0.982014,0.500000,0.731059,0.119203,0.999089,0.880797,0.999089,0.047426,0.993307,0.268941,0.993307,0.500000


With ``return_scales=True``, the matching 20-column ``df_scales`` and ``df_cat`` are returned too. They name the PSSM dimensions (``PSSM_A`` ... ``PSSM_Y``) so that the PSSM runs through the numerical CPP workflow: :meth:`NumericalFeature.get_parts` then :meth:`CPP.run_num`:

In [3]:
dict_num, df_scales, df_cat = nf.from_pssm(pssm=pssm_dir, df_seq=df_seq, return_scales=True)
aa.display_df(df_cat, n_rows=10, show_shape=True)

DataFrame shape: (20, 5)


,scale_id,category,subcategory,scale_name,scale_description
1,PSSM_A,Nonpolar,Nonpolar,PSSM A,PSSM log-odds s...or amino acid A
2,PSSM_C,Polar,Polar,PSSM C,PSSM log-odds s...or amino acid C
3,PSSM_D,Negative,Negative,PSSM D,PSSM log-odds s...or amino acid D
4,PSSM_E,Negative,Negative,PSSM E,PSSM log-odds s...or amino acid E
5,PSSM_F,Aromatic,Aromatic,PSSM F,PSSM log-odds s...or amino acid F
6,PSSM_G,Nonpolar,Nonpolar,PSSM G,PSSM log-odds s...or amino acid G
7,PSSM_H,Positive,Positive,PSSM H,PSSM log-odds s...or amino acid H
8,PSSM_I,Nonpolar,Nonpolar,PSSM I,PSSM log-odds s...or amino acid I
9,PSSM_K,Positive,Positive,PSSM K,PSSM log-odds s...or amino acid K
10,PSSM_L,Nonpolar,Nonpolar,PSSM L,PSSM log-odds s...or amino acid L


In [4]:
labels = df_seq["label"].to_list()
df_parts, dict_num_parts = nf.get_parts(df_seq=df_seq, dict_num=dict_num)
cpp = aa.CPP(df_parts=df_parts, df_scales=df_scales, df_cat=df_cat, verbose=False)
df_feat = cpp.run_num(dict_num_parts=dict_num_parts, labels=labels, n_filter=50, n_jobs=1)
aa.display_df(df_feat, n_rows=10, show_shape=True)

DataFrame shape: (50, 13)


,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions
1,"TMD-Pattern(C,9,12)-PSSM_H",Positive,Positive,PSSM H,PSSM log-odds s...or amino acid H,0.500000,0.609000,0.609000,0.137000,0.216000,0.000157,0.026393,"19,22"
2,"TMD-Pattern(N,12,15)-PSSM_H",Positive,Positive,PSSM H,PSSM log-odds s...or amino acid H,0.500000,0.609000,0.609000,0.137000,0.216000,0.000157,0.026393,"22,25"
3,"JMD_N_TMD_N-Pat...n(C,3,7)-PSSM_P",Nonpolar,Nonpolar,PSSM P,PSSM log-odds s...or amino acid P,0.500000,0.407000,0.407000,0.100000,0.134000,0.000157,0.026393,"14,18"
4,"TMD-Pattern(N,6,10)-PSSM_P",Nonpolar,Nonpolar,PSSM P,PSSM log-odds s...or amino acid P,0.500000,0.407000,0.407000,0.100000,0.134000,0.000157,0.026393,"16,20"
5,"JMD_N_TMD_N-Seg...t(12,15)-PSSM_F",Aromatic,Aromatic,PSSM F,PSSM log-odds s...or amino acid F,0.470000,0.588000,0.588000,0.104000,0.264000,0.000381,0.026393,"15,16"
6,"TMD-Pattern(N,7,10,14)-PSSM_F",Aromatic,Aromatic,PSSM F,PSSM log-odds s...or amino acid F,0.460000,0.408000,0.408000,0.103000,0.207000,0.000507,0.026393,"17,20,24"
7,"TMD-Segment(10,14)-PSSM_P",Nonpolar,Nonpolar,PSSM P,PSSM log-odds s...or amino acid P,0.460000,0.380000,-0.380000,0.191000,0.081000,0.000507,0.026393,"23,24"
8,"JMD_N_TMD_N-Pat...,5,9,13)-PSSM_P",Nonpolar,Nonpolar,PSSM P,PSSM log-odds s...or amino acid P,0.460000,0.159000,-0.159000,0.068000,0.060000,0.000507,0.026393,"2,5,9,13"
9,"JMD_N_TMD_N-Pat...C,1,5,8)-PSSM_W",Aromatic,Aromatic,PSSM W,PSSM log-odds s...or amino acid W,0.450000,0.264000,-0.264000,0.126000,0.107000,0.000670,0.026422,"13,16,20"
10,"TMD-Pattern(N,5,8,12)-PSSM_W",Aromatic,Aromatic,PSSM W,PSSM log-odds s...or amino acid W,0.450000,0.264000,-0.264000,0.126000,0.107000,0.000670,0.026422,"15,18,22"


**Further parameters.** ``values`` selects the PSSM block: ``'log_odds'`` (default) or ``'frequencies'`` (the weighted observed percentages). ``normalize`` maps log-odds via the sigmoid and percentages via division by 100 onto ``[0, 1]``; set it to ``False`` for the raw values. ``pssm`` also accepts a path to a single ``.pssm`` file (the file stem becomes the entry) or a dict mapping entries to file paths or precomputed ``(L, 20)`` arrays (already in canonical column order):

In [5]:
dict_freq = nf.from_pssm(pssm=pssm_dir, values="frequencies", normalize=False)
df_freq = pd.DataFrame(dict_freq[entry], columns=aa.utils.LIST_CANONICAL_AA)
aa.display_df(df_freq, n_rows=10, show_shape=True)

DataFrame shape: (87, 20)


,A,C,D,E,F,G,H,I,K,L,M,N,P,Q,R,S,T,V,W,Y
1,20.000000,31.000000,37.000000,16.000000,22.000000,32.000000,21.000000,17.000000,9.000000,18.000000,3.000000,7.000000,35.000000,25.000000,14.000000,2.000000,34.000000,25.000000,33.000000,11.000000
2,36.000000,16.000000,38.000000,36.000000,5.000000,14.000000,3.000000,18.000000,7.000000,31.000000,18.000000,14.000000,27.000000,13.000000,29.000000,19.000000,13.000000,26.000000,9.000000,22.000000
3,33.000000,31.000000,0.000000,31.000000,18.000000,26.000000,18.000000,28.000000,31.000000,11.000000,22.000000,32.000000,20.000000,31.000000,7.000000,22.000000,1.000000,4.000000,5.000000,9.000000
4,39.000000,39.000000,16.000000,1.000000,11.000000,9.000000,32.000000,2.000000,11.000000,34.000000,36.000000,11.000000,17.000000,34.000000,8.000000,26.000000,5.000000,31.000000,22.000000,20.000000
5,37.000000,19.000000,27.000000,6.000000,14.000000,15.000000,9.000000,12.000000,25.000000,27.000000,24.000000,19.000000,38.000000,17.000000,6.000000,3.000000,13.000000,38.000000,4.000000,13.000000
6,5.000000,28.000000,8.000000,32.000000,30.000000,23.000000,21.000000,7.000000,34.000000,18.000000,0.000000,28.000000,19.000000,12.000000,18.000000,28.000000,26.000000,25.000000,17.000000,12.000000
7,31.000000,39.000000,37.000000,17.000000,38.000000,13.000000,24.000000,23.000000,0.000000,11.000000,6.000000,11.000000,16.000000,23.000000,6.000000,19.000000,18.000000,3.000000,31.000000,2.000000
8,14.000000,21.000000,35.000000,12.000000,11.000000,22.000000,12.000000,4.000000,26.000000,26.000000,4.000000,0.000000,9.000000,5.000000,33.000000,26.000000,29.000000,30.000000,29.000000,37.000000
9,25.000000,36.000000,20.000000,39.000000,17.000000,37.000000,7.000000,6.000000,1.000000,21.000000,11.000000,24.000000,38.000000,10.000000,11.000000,39.000000,6.000000,29.000000,35.000000,3.000000
10,0.000000,1.000000,9.000000,2.000000,23.000000,33.000000,35.000000,6.000000,7.000000,29.000000,14.000000,37.000000,28.000000,4.000000,37.000000,34.000000,1.000000,12.000000,7.000000,11.000000


In [6]:
pssm = {"file_entry": str(first_file),
        "array_entry": np.zeros((5, 20))}   # raw log-odds of 0 everywhere
dict_mixed = nf.from_pssm(pssm=pssm, values="log_odds", normalize=True, return_scales=False)
df_mixed = pd.DataFrame(dict_mixed["array_entry"], columns=aa.utils.LIST_CANONICAL_AA)   # sigmoid(0) = 0.5
aa.display_df(df_mixed, n_rows=10, show_shape=True)

dict_one = nf.from_pssm(pssm=first_file)   # single '.pssm' file: the file stem becomes the entry
print(list(dict_one), dict_one[first_file.stem].shape)

DataFrame shape: (5, 20)


,A,C,D,E,F,G,H,I,K,L,M,N,P,Q,R,S,T,V,W,Y
1,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000
2,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000
3,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000
4,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000
5,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000


['O43914'] (113, 20)
